# Classical ML Models

## Goal

Evaluate classical machine learning models for privacy sensitive prompt detection.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

This notebook trains and evaluates the following models:
- Logistic Regression
- Naive Bayes
- Linear SVM



In [ ]:
# Imports
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.svm import LinearSVC 
from pathlib import Path
import numpy as np
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
import time
import os
import json

## Load Frozen Data Splits
Train, validation, and test sets generated by 03_feature_engineering.ipynb.

In [4]:
FEATURE_DIR = Path('../../feature_matrices')
train_features = load_npz(FEATURE_DIR / 'X_train_combined.npz')
val_features = load_npz(FEATURE_DIR / 'X_val_combined.npz')
test_features = load_npz(FEATURE_DIR / 'X_test_combined.npz')

train_labels = np.load(FEATURE_DIR / 'y_train.npy')
val_labels = np.load(FEATURE_DIR / 'y_val.npy')
test_labels = np.load(FEATURE_DIR / 'y_test.npy')

In [5]:
# data set verification sanity check
print(train_features.shape)
print(val_features.shape)
print(test_features.shape)

(260338, 50018)
(32542, 50018)
(32543, 50018)


## Baseline Model: Always Predict PII

As a simple baseline, we evaluate a model that predicts every prompt as PII. Since the dataset has more sensitive examples than safe examples, this baseline helps show whether the trained models are learning beyond the class imbalance.

In [6]:
# create array of 1s to serve as the baseline
baseline_predictions = np.ones(len(val_labels), dtype=int)
# zero_division set to 0 to hide warning due to zero examples being predicted as safe
print(classification_report(val_labels, baseline_predictions, zero_division=0))

cm = confusion_matrix(val_labels, baseline_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10616
           1       0.67      1.00      0.81     21926

    accuracy                           0.67     32542
   macro avg       0.34      0.50      0.40     32542
weighted avg       0.45      0.67      0.54     32542



,Predicted Safe,Predicted PII
Actual Safe,0,10616
Actual PII,0,21926


## Logistic Regression

Logistic Regression is a common baseline for text classification because it performs well on sparse TF-IDF representations and produces interpretable feature weights. However, the model learns a linear decision boundary and may struggle to capture complex relationships between terms.

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

https://www.geeksforgeeks.org/machine-learning/understanding-logistic-regression/

The model computes a weighted sum of the input features:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

The sigmoid function maps this score to the range [0, 1]:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

The resulting value is interpreted as the probability that a prompt belongs to the privacy sensitive class:

$$
P(y=1|x) = \sigma(z)
$$

### Tuning

In [ ]:
# leaving outside for if combined comparison is made later
results = []
save_path = 'classical_tuning_results.csv'

# load data from csv
if os.path.exists(save_path):
    results_df = pd.read_csv(save_path)
    results = results_df.to_dict('records')
    completed_keys = set(results_df['config_key'])
    print(f'Loaded {len(results)} saved results from CSV.')

# helper function to make keys
def make_key(model_name, params):
    return f"{model_name}|{json.dumps(params, sort_keys=True)}"

# helper function for tuning the classical models

def run_search(model_name, base_model, param_grid):
    configs = list(ParameterGrid(param_grid))

    print(f'\n----- {model_name} tuning started -----')
    print(f'Total configs for {model_name}: {len(configs)}')

    for i, params in enumerate(configs, start=1):
        # if already in csv, skip to save time
        key = make_key(model_name, params)

        if key in completed_keys:
            print(f'Skipping {model_name} {i}/{len(configs)}: already saved')
            continue

        print('\n' + '-' * 60)
        print(f'{model_name} progress: {i}/{len(configs)}')
        print(f'Params: {params}')

        # clone the base model and use this iteration's params
        model = clone(base_model)
        model.set_params(**params)
        
        # keep track of time to find bottlenecks
        start = time.perf_counter()

        # train model
        model.fit(train_features, train_labels)
        preds = model.predict(val_features)

        elapsed = (time.perf_counter() - start) / 60

        row = {
            'model': model_name,
            'params': str(params),
            **params,
            'accuracy': accuracy_score(val_labels, preds),
            'precision': precision_score(val_labels, preds),
            'recall': recall_score(val_labels, preds),
            'f1': f1_score(val_labels, preds),
            'time_min': elapsed
        }

        results.append(row)
        completed_keys.add(key)
        # save to csv so that this doesn't need to be run everytime
        results_df = pd.DataFrame(results)
        results_df = results_df.drop_duplicates(subset=["config_key"], keep="last")
        results_df.to_csv(save_path, index=False)

        results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
        # best = results_df.iloc[0]

        # progress tracking (made the mistake of not doing this last time)
        print(
            f"Finished in {elapsed:.2f} min\n"
            f"F1:        {row['f1']:.4f}\n"
            f"Precision: {row['precision']:.4f}\n"
            f"Recall:    {row['recall']:.4f}"
        )

        # print(f"Best overall so far: {best['model']} | F1={best['f1']:.4f}")

    print(f'\n----- {model_name} tuning complete -----')

    model_results = (pd.DataFrame(results).query('model == @model_name').sort_values('f1', ascending=False))

    print(f'\nTop results for {model_name}:')
    display(model_results.head(10))

In [ ]:
# saga with elastic was taking too long so the solver has been switched over to liblinear
run_search(
    model_name='Logistic Regression',
    base_model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=42),
    param_grid={'C': [0.1, 1, 2.5, 2.6, 2.65, 2.7, 2.75, 2.8, 2.85, 2.9, 2.95, 3, 3.25, 3.5, 5, 10], 'l1_ratio': [0.0, 1.0]}
)


----- Logistic Regression tuning started -----
Total configs for Logistic Regression: 26

------------------------------------------------------------
Logistic Regression progress: 1/26
Params: {'C': 0.1, 'l1_ratio': 0.0}
Finished in 0.15 min
F1:        0.8476
Precision: 0.8061
Recall:    0.8936
Best overall so far: Logistic Regression | F1=0.8476

------------------------------------------------------------
Logistic Regression progress: 2/26
Params: {'C': 0.1, 'l1_ratio': 1.0}
Finished in 0.27 min
F1:        0.8356
Precision: 0.8034
Recall:    0.8705
Best overall so far: Logistic Regression | F1=0.8476

------------------------------------------------------------
Logistic Regression progress: 3/26
Params: {'C': 1, 'l1_ratio': 0.0}
Finished in 0.30 min
F1:        0.8703
Precision: 0.8536
Recall:    0.8877
Best overall so far: Logistic Regression | F1=0.8703

------------------------------------------------------------
Logistic Regression progress: 4/26
Params: {'C': 1, 'l1_ratio': 1.0

,model,params,C,l1_ratio,accuracy,precision,recall,f1,time_min
7,Logistic Regression,"{'C': 2.75, 'l1_ratio': 1.0}",2.75,1.0,0.830004,0.868869,0.880598,0.874694,0.920900
5,Logistic Regression,"{'C': 2.5, 'l1_ratio': 1.0}",2.50,1.0,0.829974,0.868763,0.880690,0.874686,0.937029
9,Logistic Regression,"{'C': 2.8, 'l1_ratio': 1.0}",2.80,1.0,0.829758,0.868821,0.880234,0.874490,0.956742
13,Logistic Regression,"{'C': 2.9, 'l1_ratio': 1.0}",2.90,1.0,0.829666,0.868505,0.880507,0.874465,0.951726
11,Logistic Regression,"{'C': 2.85, 'l1_ratio': 1.0}",2.85,1.0,0.829697,0.868677,0.880325,0.874462,0.970630
15,Logistic Regression,"{'C': 2.95, 'l1_ratio': 1.0}",2.95,1.0,0.829205,0.868019,0.880370,0.874151,1.036657
17,Logistic Regression,"{'C': 3, 'l1_ratio': 1.0}",3.00,1.0,0.829113,0.868166,0.880005,0.874046,0.941514
19,Logistic Regression,"{'C': 3.25, 'l1_ratio': 1.0}",3.25,1.0,0.828714,0.868156,0.879321,0.873703,0.992891
21,Logistic Regression,"{'C': 3.5, 'l1_ratio': 1.0}",3.50,1.0,0.828560,0.868358,0.878774,0.873535,1.034265
18,Logistic Regression,"{'C': 3.25, 'l1_ratio': 0.0}",3.25,0.0,0.826808,0.862612,0.883700,0.873029,0.410181


In [ ]:
run_search(
    model_name='Logistic Regression ElasticNet',
    base_model=LogisticRegression(solver='saga', max_iter=500, tol=1e-3, random_state=42),
    param_grid={'C': [2.5, 2.75, 3, 3.25, 3.5], 'l1_ratio': [0.25, 0.5, 0.75]}
)


----- Logistic Regression ElasticNet tuning started -----
Total configs for Logistic Regression ElasticNet: 15

------------------------------------------------------------
Logistic Regression ElasticNet progress: 1/15
Params: {'C': 2.5, 'l1_ratio': 0.25}


### Training

Fit the logistic regression model using the engineered combined feature representation of the training data.
Use set seed (42) to ensure reproducible results across runs / team members. 

In [ ]:
# reused seed used in data split script
lr = LogisticRegression(C=2.75, l1_ratio=1, random_state=42, max_iter=1000, solver='liblinear')
lr.fit(train_features, train_labels)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",3
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'

### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

For this project, special attention should go towards false negatives since they represent privacy sensitive prompts incorrectly classified as safe.

[![Confusion Matrix](https://i0.wp.com/statisticsbyjim.com/wp-content/uploads/2025/05/confusion_matrix-1.png?fit=550%2C450&ssl=1)](https://statisticsbyjim.com/glossary/confusion-matrix/)

In [ ]:
lr_predictions = lr.predict(val_features)
print(classification_report(val_labels, lr_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, lr_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.74      0.72      0.73     10616
         PII       0.87      0.88      0.87     21926

    accuracy                           0.83     32542
   macro avg       0.81      0.80      0.80     32542
weighted avg       0.83      0.83      0.83     32542



,Predicted Safe,Predicted PII
Actual Safe,7686,2930
Actual PII,2631,19295


## Naive Bayes

Multinomial Naive Bayes is a probabilistic classification algorithm also commonly used for text classification.

The model estimates the probability that a prompt belongs to each class based on the observed features and predicts the class with the highest posterior probability.

https://scikit-learn.org/stable/modules/naive_bayes.html

https://scikit-learn.org/stable/api/sklearn.naive_bayes.html

Bayes' Theorem:

$$
P(y|x)=\frac{P(x|y)P(y)}{P(x)}
$$

Naive Bayes assumes that features are conditionally independent given the class label.

$$
P(x_1,x_2,\ldots,x_n|y)
=
\prod_{i=1}^{n} P(x_i|y)
$$

In other words, once the model knows whether a prompt belongs to the Safe or PII class, it treats each feature as contributing independently to the final prediction. While this assumption is rarely true for real-world text data, it greatly simplifies computation and often performs surprisingly well for text classification tasks.

### Tuning

In [ ]:
run_search(
    model_name='MultinomialNB',
    base_model=MultinomialNB(),
    param_grid={'alpha': [0.01, 0.05, 0.1, 0.5, 1.0]}
)

run_search(
    model_name='BernoulliNB',
    base_model=BernoulliNB(binarize=0.0),
    param_grid={'alpha': [0.01, 0.05, 0.1, 0.5, 1.0]}
)


----- MultinomialNB tuning started -----
Total configs for MultinomialNB: 5

------------------------------------------------------------
MultinomialNB progress: 1/5
Params: {'alpha': 0.01}
Finished in 0.01 min
F1:        0.8165
Precision: 0.8082
Recall:    0.8250
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
MultinomialNB progress: 2/5
Params: {'alpha': 0.05}
Finished in 0.00 min
F1:        0.8160
Precision: 0.8081
Recall:    0.8240
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
MultinomialNB progress: 3/5
Params: {'alpha': 0.1}
Finished in 0.00 min
F1:        0.8156
Precision: 0.8077
Recall:    0.8236
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
MultinomialNB progress: 4/5
Params: {'alpha': 0.5}
Finished in 0.00 min
F1:        0.8140
Precision: 0.8039
Recall:    0.8244
Best overa

,model,params,C,class_weight,l1_ratio,accuracy,precision,recall,f1,time_min,alpha
16,MultinomialNB,{'alpha': 0.01},NaN,NaN,NaN,0.750169,0.808194,0.825002,0.816512,0.013214,0.01
17,MultinomialNB,{'alpha': 0.05},NaN,NaN,NaN,0.749554,0.808050,0.824045,0.815969,0.001241,0.05
18,MultinomialNB,{'alpha': 0.1},NaN,NaN,NaN,0.749063,0.807747,0.823588,0.815591,0.001445,0.10
20,MultinomialNB,{'alpha': 1.0},NaN,NaN,NaN,0.744822,0.798388,0.831159,0.814444,0.001848,1.00
19,MultinomialNB,{'alpha': 0.5},NaN,NaN,NaN,0.746174,0.803878,0.824409,0.814014,0.001517,0.50



----- BernoulliNB tuning started -----
Total configs for BernoulliNB: 5

------------------------------------------------------------
BernoulliNB progress: 1/5
Params: {'alpha': 0.01}
Finished in 0.00 min
F1:        0.7877
Precision: 0.8570
Recall:    0.7288
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
BernoulliNB progress: 2/5
Params: {'alpha': 0.05}
Finished in 0.00 min
F1:        0.7866
Precision: 0.8570
Recall:    0.7269
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
BernoulliNB progress: 3/5
Params: {'alpha': 0.1}
Finished in 0.00 min
F1:        0.7854
Precision: 0.8568
Recall:    0.7250
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
BernoulliNB progress: 4/5
Params: {'alpha': 0.5}
Finished in 0.00 min
F1:        0.7833
Precision: 0.8569
Recall:    0.7213
Best overall so far: L

,model,params,C,class_weight,l1_ratio,accuracy,precision,recall,f1,time_min,alpha
21,BernoulliNB,{'alpha': 0.01},NaN,NaN,NaN,0.735327,0.857013,0.728769,0.787706,0.003410,0.01
22,BernoulliNB,{'alpha': 0.05},NaN,NaN,NaN,0.734313,0.857028,0.726945,0.786645,0.002675,0.05
23,BernoulliNB,{'alpha': 0.1},NaN,NaN,NaN,0.733114,0.856843,0.725030,0.785444,0.002876,0.10
24,BernoulliNB,{'alpha': 0.5},NaN,NaN,NaN,0.731025,0.856856,0.721290,0.783250,0.002813,0.50
25,BernoulliNB,{'alpha': 1.0},NaN,NaN,NaN,0.729642,0.855965,0.719876,0.782044,0.003067,1.00


### Training

Fit the Naive Bayes model using the engineered combined feature representation of the training data.

In [ ]:
nb = MultinomialNB(alpha=0.01)
nb.fit(train_features, train_labels)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.01
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[ 84930.,175408.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-1.12,-0.39]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 50018)","[[ 383.17, 151.52, 1.81,..., 2921. , 529. , 528. ], [ 1306.95, 596.55, 5.14,...,12122. , 6996. , 3669. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 50018)","[[ -7.19, -8.12,-12.54,..., -5.16, -6.87, -6.87], [ -6.87, -7.65,-12.4 ,..., -4.64, -5.19, -5.83]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,50018


### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

In [ ]:
nb_predictions = nb.predict(val_features)
print(classification_report(val_labels, nb_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, nb_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.62      0.60      0.61     10616
         PII       0.81      0.83      0.82     21926

    accuracy                           0.75     32542
   macro avg       0.72      0.71      0.71     32542
weighted avg       0.75      0.75      0.75     32542



,Predicted Safe,Predicted PII
Actual Safe,6323,4293
Actual PII,3837,18089


## Linear Support Vector Machine (SVM)

A Linear Support Vector Machine is a classification model that tries to find a decision boundary separating the Safe and PII classes with the largest possible margin.

https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html

https://www.geeksforgeeks.org/machine-learning/support-vector-machine-algorithm/

For a linear decision boundary, the model computes:

$$
f(x) = w^T x + b
$$

- Class 0: f(x) > 0
- Class 1: f(x) < 0

Predictions are based on which side of the decision boundary the example falls on.


### Tuning

In [ ]:
run_search(
    model_name='LinearSVC',
    base_model=LinearSVC(dual='auto', max_iter=3000, tol=1e-3, random_state=42),
    param_grid={'C': [0.1, 1, 3, 10], 'class_weight': [None, 'balanced']}
)


----- LinearSVC tuning started -----
Total configs for LinearSVC: 8

------------------------------------------------------------
LinearSVC progress: 1/8
Params: {'C': 0.1, 'class_weight': None}
Finished in 0.67 min
F1:        0.8707
Precision: 0.8549
Recall:    0.8870
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
LinearSVC progress: 2/8
Params: {'C': 0.1, 'class_weight': 'balanced'}
Finished in 0.87 min
F1:        0.8358
Precision: 0.9189
Recall:    0.7665
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
LinearSVC progress: 3/8
Params: {'C': 1, 'class_weight': None}
Finished in 1.66 min
F1:        0.8717
Precision: 0.8667
Recall:    0.8768
Best overall so far: Logistic Regression | F1=0.8740

------------------------------------------------------------
LinearSVC progress: 4/8
Params: {'C': 1, 'class_weight': 'balanced'}
Finished in 1.87 min
F1:       

,model,params,C,class_weight,l1_ratio,accuracy,precision,recall,f1,time_min,alpha
28,LinearSVC,"{'C': 1, 'class_weight': None}",1.0,NaN,NaN,0.826132,0.866727,0.876767,0.871718,1.657867,NaN
26,LinearSVC,"{'C': 0.1, 'class_weight': None}",0.1,NaN,NaN,0.822445,0.854901,0.887029,0.870669,0.673273,NaN
30,LinearSVC,"{'C': 3, 'class_weight': None}",3.0,NaN,NaN,0.821615,0.864943,0.871294,0.868107,1.573821,NaN
32,LinearSVC,"{'C': 10, 'class_weight': None}",10.0,NaN,NaN,0.814240,0.860391,0.864590,0.862485,2.404436,NaN
29,LinearSVC,"{'C': 1, 'class_weight': 'balanced'}",1.0,balanced,NaN,0.814148,0.904350,0.809815,0.854475,1.870384,NaN
31,LinearSVC,"{'C': 3, 'class_weight': 'balanced'}",3.0,balanced,NaN,0.810491,0.894231,0.815151,0.852862,2.071985,NaN
33,LinearSVC,"{'C': 10, 'class_weight': 'balanced'}",10.0,balanced,NaN,0.806158,0.884150,0.819712,0.850712,3.068892,NaN
27,LinearSVC,"{'C': 0.1, 'class_weight': 'balanced'}",0.1,balanced,NaN,0.797124,0.918917,0.766533,0.835836,0.874101,NaN


### Training

Fit the Linear Support Vector Machine model using the engineered combined feature representation of the training data.

In [ ]:
# reused seed used in data split script
svm = LinearSVC(C=1, random_state=42)
svm.fit(train_features, train_labels)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjus

### Evaluation

Generate predictions on the validation set for model evaluation, then evaluate model performance using a classification report and confusion matrix.

In [ ]:
svm_predictions = svm.predict(val_features)
print(classification_report(val_labels, svm_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(val_labels, svm_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.74      0.72      0.73     10616
         PII       0.87      0.88      0.87     21926

    accuracy                           0.83     32542
   macro avg       0.80      0.80      0.80     32542
weighted avg       0.83      0.83      0.83     32542



,Predicted Safe,Predicted PII
Actual Safe,7661,2955
Actual PII,2706,19220


# Final Test Evaluation

After comparing the models on the validation set, the final model configurations are evaluated on a held out test set. These results will provide an unbiased estimate of each model's performance on unseen data.

#### Baseline

In [ ]:
baseline_test_predictions = np.ones(len(test_labels), dtype=int)
# zero_division set to 0 to hide warning due to zero examples being predicted as safe
print(classification_report(test_labels, baseline_test_predictions, zero_division=0))

cm = confusion_matrix(test_labels, baseline_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     10617
           1       0.67      1.00      0.81     21926

    accuracy                           0.67     32543
   macro avg       0.34      0.50      0.40     32543
weighted avg       0.45      0.67      0.54     32543



,Predicted Safe,Predicted PII
Actual Safe,0,10617
Actual PII,0,21926


#### Logistic Regression

In [ ]:
lr_test_predictions = lr.predict(test_features)
print(classification_report(test_labels, lr_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, lr_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.76      0.73      0.74     10617
         PII       0.87      0.88      0.88     21926

    accuracy                           0.84     32543
   macro avg       0.81      0.81      0.81     32543
weighted avg       0.83      0.84      0.84     32543



,Predicted Safe,Predicted PII
Actual Safe,7798,2819
Actual PII,2524,19402


#### Naive Bayes

In [ ]:
nb_test_predictions = nb.predict(test_features)
print(classification_report(test_labels, nb_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, nb_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.63      0.59      0.61     10617
         PII       0.81      0.83      0.82     21926

    accuracy                           0.75     32543
   macro avg       0.72      0.71      0.71     32543
weighted avg       0.75      0.75      0.75     32543



,Predicted Safe,Predicted PII
Actual Safe,6317,4300
Actual PII,3758,18168


#### Linear SVM

In [ ]:
svm_test_predictions = svm.predict(test_features)
print(classification_report(test_labels, svm_test_predictions, target_names=['Safe', 'PII']))
cm = confusion_matrix(test_labels, svm_test_predictions)

pd.DataFrame(cm, index=['Actual Safe', 'Actual PII'], columns=['Predicted Safe', 'Predicted PII'])

              precision    recall  f1-score   support

        Safe       0.75      0.73      0.74     10617
         PII       0.87      0.88      0.88     21926

    accuracy                           0.83     32543
   macro avg       0.81      0.80      0.81     32543
weighted avg       0.83      0.83      0.83     32543



,Predicted Safe,Predicted PII
Actual Safe,7728,2889
Actual PII,2593,19333


## Results Summary Chart

In [ ]:
predictions = {
    'Baseline': baseline_test_predictions,
    'Logistic Regression': lr_test_predictions,
    'Naive Bayes': nb_test_predictions,
    'Linear SVM': svm_test_predictions
}

comparison_results = []

for model, p in predictions.items():
    comparison_results.append({
        'Model': model,
        'Accuracy': accuracy_score(test_labels, p),
        'PII Precision': precision_score(test_labels, p, zero_division=0),
        'PII Recall': recall_score(test_labels, p, zero_division=0),
        'PII F1': f1_score(test_labels, p, zero_division=0)
    })

comparison_results = pd.DataFrame(comparison_results).round(4)

comparison_results

,Model,Accuracy,PII Precision,PII Recall,PII F1
0,Baseline,0.6738,0.6738,1.0000,0.8051
1,Logistic Regression,0.8358,0.8731,0.8849,0.8790
2,Naive Bayes,0.7524,0.8086,0.8286,0.8185
3,Linear SVM,0.8315,0.8700,0.8817,0.8758
